### TP INF 372 : Analyse de la Consommation Data

| Nom Complet | Matricule |
| :--- | :--- |
| KAMDOM NGUETCHESSI MERVEILLE PRISCA | 23V2060 |
| YOKO AYMERICK JEFFREY | 23U2983 |
| KENNE YEMTSA LOWEL RAYANN | 23U2477 |

---

# Composition et Nettoyage du Jeu de Données

Ce notebook constitue la première étape de notre démarche. Son objectif est de collecter les multiples fichiers CSV bruts issus de différents relevés de consommation internet, de les nettoyer et de les regrouper dans un dataset unique consolidé.

### L'Intuition Derrière cette Étape
La donnée brute est rarement parfaite. Lors de nos premières explorations, nous avons découvert que certains fichiers n'avaient pas d'en-têtes, d'autres avaient des colonnes décalées, et surtout, certains contenaient des **valeurs négatives** (probablement des erreurs de saisie ou d'exportation). Sans un nettoyage rigoureux à la source, tout modèle de Machine Learning entraîné par la suite serait biaisé.

### Préparation de l'environnement
Nous utilisons `pandas` pour la manipulation de données et `glob` pour parcourir les dossiers.

In [ ]:
!pip install pandas numpy matplotlib scikit-learn


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import pathlib

RAW_DATA_DIR = "../../data/raw"
OUTPUT_FILE = "../../data/processed/composed_dataset.csv"

EXPECTED_COLUMNS = [
    'date', 'heure', 'jour_semaine', 'periode_journee', 'statut_connexion', 
    'duree_minutes', 'download_Mo', 'upload_Mo', 'total_Mo', 'session_id'
]

## 1. Chargement et Nettoyage Individuel

### Pourquoi filtrer spécifiquement certains fichiers ?
Nous avons appris que certains jeux de données (comme celui marqué 'TUME') présentaient des incohérences de langue ou des formats non standard qui nuisaient à la cohérence globale du dataset composé. Nous avons donc choisi de les exclure pour maintenir une qualité de données élevée.

In [2]:
csv_files = glob.glob(os.path.join(RAW_DATA_DIR, "*.csv"))
dfs = []

print(f"Fichiers trouvés total : {len(csv_files)}")

for filepath in csv_files:
    filename = os.path.basename(filepath)
    
    # Exclusion des fichiers identifiés comme problématiques lors de l'investigation initiale
    if "TUME_FLIZOT_FONDZENYUY_21T2625".lower() in filename.lower():
        continue
        
    try:
        df = pd.read_csv(filepath)
        
        # Intuition : Si la première colonne ressemble à une date, c'est que le header est absent.
        if str(df.columns[0]).startswith('202'):
            df = pd.read_csv(filepath, header=None)
            
        # On ne garde que les fichiers respectant exactement le format attendu (10 colonnes)
        if len(df.columns) != 10:
            continue
            
        df.columns = EXPECTED_COLUMNS
        
        # Élimination des lignes vides ou incomplètes
        df = df.dropna(how='any')
        
        # --- ÉTAPE CRUCIALE : Suppression des valeurs négatives ---
        # Pourquoi ? Une durée ou un volume de données ne peut physiologiquement pas être négatif.
        # Ces lignes sont des bruits de mesure qui fausseraient les moyennes du modèle.
        num_cols = ['duree_minutes', 'download_Mo', 'upload_Mo', 'total_Mo']
        for col in num_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df = df[df[col] >= 0]
            
        df = df.dropna()
        
        if not df.empty:
            dfs.append(df)
            
    except Exception as e:
        print(f"Fichier ignoré : {filename} ({e})")

print(f"Fichiers valides retenus : {len(dfs)}")

Fichiers trouvés total : 45
Fichiers valides retenus : 43


## 2. Consolidation et Tri

### Intuition
La navigation internet est une série temporelle. Pour que nos futurs modèles puissent potentiellement apprendre des séquences, il est impératif que le dataset soit trié chronologiquement.

In [3]:
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df = combined_df.sort_values(by=['date', 'heure']).reset_index(drop=True)
    
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    combined_df.to_csv(OUTPUT_FILE, index=False)
    
    print("--- Résumé du Dataset Consolidé ---")
    print(f"Nombre total de sessions : {len(combined_df):,}")
    display(combined_df.head())
else:
    print("Erreur : Aucune donnée extraite.")

--- Résumé du Dataset Consolidé ---
Nombre total de sessions : 86,651


,date,heure,jour_semaine,periode_journee,statut_connexion,duree_minutes,download_Mo,upload_Mo,total_Mo,session_id
0,2026-04-21,00:02:15,Mardi,Nuit profonde (00h-05h),Connecté,1.0,0.1873,0.0380,0.2253,S20260420_211411
1,2026-04-18,18:05:24,Samedi,Soirée (18h-21h),Connecté,1.0,0.1632,0.1016,0.2648,S20260418_180424
2,2026-04-18,18:06:27,Samedi,Soirée (18h-21h),Déconnecté,1.0,0.2574,0.2267,0.4841,S20260418_180424
3,2026-04-18,18:07:27,Samedi,Soirée (18h-21h),Connecté,1.0,0.0774,0.1614,0.2388,S20260418_180424
4,2026-04-18,18:08:27,Samedi,Soirée (18h-21h),Connecté,1.0,0.0513,0.0733,0.1246,S20260418_180424


## Conclusion du Notebook 01
Nous avons réussi à passer d'une multitude de fichiers hétérogènes à une base de données unique et saine. 
**Résultat attendu** : Une base fiable de ~86 000 lignes prêtes pour l'analyse sans risque d'erreurs mathématiques dues à des valeurs négatives.